# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KhanBuilds/Rayanflyrank/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

**Lane 2: Refresh / Content Opportunity Scoring.** Two findings from the FlyRank research paper put under
the same standard I hold my own work to, then my own model attacked with the tools I used on theirs.

Continues from `w05_model.ipynb` (ML-08). The rule I am applying to both, from
`skills/writing-honest-claims`: *where does the label come from, what does the validation design actually
test, and would the number survive a grouped or time split?*

**A note on posture.** The paper under review is the work of the team hosting this internship, and its
Methodology page is more careful than most published SEO research — it names its confounders, flags its
own unstable buckets, and says plainly that the study is observational. My questions below are aimed at
making specific claims stronger, not at scoring points, and where the paper polices itself well I say so.
The two places I do push hard are places where **the paper contradicts either itself or the dataset it
shipped with**, and both are checkable in code rather than matters of opinion.

In [1]:
# Setup: work from the repo root so the starter CSV loads on Colab AND from a fresh local clone.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/KhanBuilds/Rayanflyrank"
REPO_DIR = "Rayanflyrank"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != os.path.dirname(os.getcwd()):
        os.chdir("..")

DATA_PATH = "data/raw/content_refresh_anonymized.csv"
assert os.path.exists(DATA_PATH), f"starter CSV not found at {DATA_PATH} - are you at the repo root?"

import json
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn
from pandas.api.types import is_numeric_dtype

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

df = pd.read_csv(DATA_PATH)
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
y = df["is_declining_label"].to_numpy()
groups = df["client_id"].to_numpy()
BASE_RATE = float(y.mean())


def precision_at_k(labels, scores, k: int) -> float:
    order = np.argsort(-np.asarray(scores, dtype=float), kind="stable")
    return float(np.asarray(labels)[order[:k]].mean())


def roc_auc(labels, scores) -> float:
    labels = np.asarray(labels)
    n_pos, n_neg = labels.sum(), (1 - labels).sum()
    if n_pos == 0 or n_neg == 0:
        return float("nan")
    ranks = pd.Series(np.asarray(scores, dtype=float)).rank().to_numpy()
    return float((ranks[labels == 1].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))


print("Working dir:", os.getcwd())
print(f"Loaded {len(df):,} rows x {df.shape[1]} columns; {df['client_id'].nunique()} clients.")
print(f"Label base rate: {BASE_RATE:.4f}")
print(f"scikit-learn {sklearn.__version__} | pandas {pd.__version__} | numpy {np.__version__}")

Working dir: C:\Users\real time\Desktop\Rayan_flyrank
Loaded 30,000 rows x 45 columns; 32 clients.
Label base rate: 0.5421
scikit-learn 1.9.0 | pandas 3.0.1 | numpy 2.4.0


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and what
would I need to see to believe it?*

Source: `docs/flyrank-seo-research-march-2026.pdf` — *The State of AI-Driven SEO*, March 2026. 341,701
content pieces, 57 brands.

---

### Finding A — "The Freshness Multiplier" (Finding #4, p.9)

> **The claim.** "At 31-90 days, freshness remains the strongest measured growth window at **7.88:1**
> [growth-to-decline ratio]." And, in the action box: *"Run a recurring refresh program for mature pages —
> **Why: Refreshing mature pages produces 3.2x health and 57x impressions in this dataset.**"*

**Where does the label come from?** `trend_direction`, computed from the 30-day-vs-previous-30-day
impression change. Freshness (`days_since_last_update`) is a CMS field, so the two do not share a
measurement window — **this is not a leakage problem**, and I want to say that clearly before raising
anything else.

**Question A1 — the metric has no base ratio next to it.** 7.88:1 is a ratio of counts, and a ratio of
counts is only interpretable against the population's overall ratio. The paper reports 74.8K growing vs
45.6K declining overall (Finding #1), so the portfolio-wide ratio is **1.64:1** — meaning the 31-90 band
is roughly **4.8× the portfolio average**, not "7.88× better than nothing". That is still a real result,
and stating it as a *lift over the base ratio* would make it both smaller and much harder to argue with.
This is the same discipline the paper applies elsewhere and the exact rule I hold myself to: no rate
without its base rate.

**Question A2 — the 57× is causal language on an observational comparison, and the paper's own
Methodology page forbids it.** Page 36 says: *"Observational study: correlations do not prove causation."*
Page 9 says refreshing *"**produces** 3.2x health and 57x impressions"*. Those two sentences cannot both
stand. **Selection bias is the specific mechanism:** nobody refreshes pages at random. The pages a team
chooses to refresh are the ones already believed to be valuable — so part of that 57× is *which pages got
picked*, not what refreshing did to them. The honest form needs no new data, only different words: *"mature
pages that were refreshed in the last 30 days show 57× the impressions of those that were not"* — and then
the causal version needs a matched or controlled design.

**Question A3 — `health` partly contains the outcome.** Health Score is defined on p.5 as impressions
(30 pts) + position (30 pts) + CTR (20) + scroll depth (20). So **60 of its 100 points are search
performance.** "Refreshing lifts health from 10.7 to 34.5" is therefore partly definitional: refreshed
pages gained impressions, and impressions are 30% of the score being reported as the effect. Reporting the
raw components separately — which the paper does elsewhere and does well — avoids this entirely.

**Credit where it is due.** The paper *flags its own unstable bucket*: it prints the 361+ ratio of 283:1
and immediately says it is "unstable: 283 growing pages versus only 1 declining." That is exactly the
small-n discipline I try to apply, and it is voluntarily disclosed against the paper's own headline.

**What I would need to believe the causal version:** a matched design — refreshed pages paired with
unrefreshed pages of similar age, position, traffic and content type, compared *after* the refresh date.
The dataset needed for that (per-page update timestamps and a post-refresh window) exists in the warehouse
release but not in the shipped snapshot.

---

### Finding B — ML Appendix, "What Predicts Growth?" (p.29)

> **The claim.** "Logistic regression (**71% holdout accuracy**) describing which sampled features separate
> growing from declining pages." Methodology (p.36): "Logistic Regression (**80/20 split**)" on "61.8K
> content pieces."

This is the finding closest to my own work, so it is the one I can test rather than merely question.

**Question B1 — 71% accuracy is reported with no base rate, so its skill is unknown.** A majority-class
classifier on the paper's own growing/declining counts (74.8K vs 45.6K) scores **62.1%** for free. If the
appendix sample carries a similar balance, 71% is roughly **9 points of skill**, not 71. On *my* slice the
majority class is 54.2%, and the cell below computes what "accuracy" is worth here. **This is the single
easiest fix in the paper: print the base rate beside the number.**

**Question B2 — an 80/20 split across 57 brands is very likely inflated, and I measured how much.** The
methodology names a random 80/20 split with no mention of grouping by brand. Rows from one brand share
hidden character, so a random split lets the model recognise *which brand* a page belongs to instead of
learning decay. **I ran exactly this experiment on the same kind of data in ML-08**: identical model,
identical features, random 80/20 vs grouped-by-client. AUC **0.777 → 0.624**. That **0.152** is not a
criticism I am importing from a textbook; it is a number from this dataset, reproduced in the cell below.

**Question B3 — accuracy is the wrong metric for the decision the paper's playbook recommends.** The
playbook tells a reader to prioritise a limited set of pages for refresh. That is capacity-constrained
ranking, where what matters is precision at the top of the list, not average correctness over 61.8K rows.
A model can be 71% accurate and still put the wrong pages in the first 50 slots — which is precisely the
defect I found in my own baseline (ML-07) and had to fix in ML-08.

**Question B4 — the coefficient chart cannot support its own caption.** The printed coefficients are
rounded to integers: Content Age 1, Days Since Update 1, Days Visible 1, Average Position 0, Word Count 0,
Impressions 0, Clicks 0, Sessions 0. Six of ten features display as "0", yet the caption concludes
"content age is the strongest negative signal ... days visible and recent impressions are among the
strongest positive signals." **The chart as printed cannot distinguish those features from each other or
from zero.** There is also a collinearity trap underneath: impressions, clicks, sessions and AI sessions
are strongly correlated, so a linear fit splits one signal across them and individual coefficients stop
being interpretable. I hit this exact problem in ML-08 — my own fit put **+1.18 on impressions and −0.62
on clicks** — and my conclusion was to refuse to interpret single coefficients and report permutation
importance instead. Same fix would apply here.

**Credit where it is due.** The appendix is explicitly labelled exploratory and secondary to the direct
aggregate comparisons, the caption says to "prioritize investigation rather than to assume a guaranteed
lever", and the paper states that no p-values or confidence intervals are reported. The framing is
appropriately hedged; my objection is to the numbers inside it, not to their status in the argument.

---

### The thing I did not expect to find: the paper and its own dataset disagree

Page 5 defines the variable both findings rest on: *"Trend Direction ... **Up: >10% growth. Down: >10%
decline. Stable: within +/-10%.**"* The data dictionary shipped with this repo defines the same field as
*"up > +20%; down < −20%"*.

**The shipped data follows the ±20% definition, not the paper's ±10%** — verified in the cell below by
reading the actual `trend_pct` ranges of each class. A reader who took the paper at its word and rebuilt
the cohorts at ±10% would produce **different cohorts from the ones the paper's numbers describe**, and
would not know why their reproduction failed. This is a documentation defect rather than an analysis
error, but it sits on the definition of the paper's central variable, so it is worth one line of errata.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one - typing sentences here breaks Run All.
# --- The definition discrepancy: which threshold does the SHIPPED data actually use? ---
print("PAPER (p.5): 'Up: >10% growth. Down: >10% decline. Stable: within +/-10%'")
print("DATA DICTIONARY: 'up > +20%; down < -20%'")
print("\nWhat the shipped data actually contains (trend_pct range within each class):")
labelled = df.dropna(subset=["trend_pct"])
for direction in ("up", "stable", "down"):
    s = labelled.loc[labelled["trend_direction"] == direction, "trend_pct"]
    print(f"  {direction:<7} n={len(s):>6,}   trend_pct from {s.min():>9.1f}% to {s.max():>9.1f}%")

up_min = labelled.loc[labelled["trend_direction"] == "up", "trend_pct"].min()
down_max = labelled.loc[labelled["trend_direction"] == "down", "trend_pct"].max()
print(f"\n  -> the boundaries are exactly +{up_min:.0f}% and {down_max:.0f}%, i.e. the +/-20% rule.")
print("  -> the paper's stated +/-10% would place every 'stable' page between -20% and -10%")
print(f"     into 'down'. That is {((labelled['trend_pct'] > -20) & (labelled['trend_pct'] < -10)).sum():,} "
      f"rows in this slice alone that would change class.")
assert abs(up_min - 20.0) < 1e-6 and abs(down_max + 20.0) < 1e-6, "the +/-20% reading does not hold"

# --- Finding A: does the freshness ratio reproduce here, and what is the base ratio? ---
print("\n" + "=" * 78)
print("FINDING A - 'growth-to-decline ratio by freshness window' (paper claims 7.88:1 at 31-90d)")
bins, labels = [0, 30, 90, 180, 360, 10**6], ["0-30", "31-90", "91-180", "181-360", "361+"]
df["freshness_band"] = pd.cut(df["days_since_last_update"], bins=bins, labels=labels, include_lowest=True)
tab = pd.crosstab(df["freshness_band"], df["trend_direction"])
for c in ("up", "down"):
    if c not in tab:
        tab[c] = 0
tab = tab[["up", "down"]].copy()
tab["ratio"] = (tab["up"] / tab["down"].replace(0, np.nan)).round(2)

overall = df["trend_direction"].value_counts()
base_ratio = overall.get("up", 0) / overall.get("down", 1)
tab["lift_vs_base"] = (tab["ratio"] / base_ratio).round(2)
print(tab.to_string())
print(f"\n  this slice's OVERALL up:down ratio = {base_ratio:.2f}:1 "
      f"({overall.get('up', 0):,} up vs {overall.get('down', 0):,} down)")
print(f"  the paper's portfolio-wide ratio    = 1.64:1 (74.8K up vs 45.6K down, Finding #1)")
print("\n  -> the raw ratios do NOT reproduce (0.37:1 here vs 7.88:1 there) - but that is a POPULATION")
print("     difference, not a contradiction: this slice is a refresh-candidate cut and is ~6x enriched")
print("     in declining pages. That is exactly why a bare ratio is not portable between populations,")
print("     and why the lift-vs-base column is the comparable quantity.")
print(f"  -> normalised, 31-90d sits at {tab.loc['31-90', 'lift_vs_base']:.2f}x this slice's base ratio, "
      f"against ~4.8x in the paper's.")
print(f"  -> small-n warning applies here too: the 31-90 band holds only "
      f"{int(tab.loc['31-90', ['up', 'down']].sum()):,} labelled pages in this slice.")
print(f"  -> and the stale story barely exists here at all: "
      f"{(df['days_since_last_update'] > 180).sum()} pages beyond 180 days, of 30,000.")

# --- Finding B1: what is 'accuracy' worth without a base rate? ---
print("\n" + "=" * 78)
print("FINDING B1 - '71% holdout accuracy', reported with no base rate")
PAPER_UP, PAPER_DOWN, PAPER_ACC = 74_800, 45_600, 0.71
majority = PAPER_UP / (PAPER_UP + PAPER_DOWN)
print(f"  majority-class accuracy on the paper's own counts "
      f"({PAPER_UP:,} up vs {PAPER_DOWN:,} down): {majority:.1%}  <- free, no model")
print(f"  so {PAPER_ACC:.0%} is about {(PAPER_ACC - majority) * 100:.1f} points of skill, not "
      f"{PAPER_ACC * 100:.0f}")
print(f"  majority-class accuracy on MY slice: {max(BASE_RATE, 1 - BASE_RATE):.1%} "
      f"(base rate {BASE_RATE:.4f})")
print("  -> any accuracy number without its base rate beside it is not yet a claim.")

PAPER (p.5): 'Up: >10% growth. Down: >10% decline. Stable: within +/-10%'
DATA DICTIONARY: 'up > +20%; down < -20%'

What the shipped data actually contains (trend_pct range within each class):
  up      n= 4,388   trend_pct from      20.0% to   44900.0%
  stable  n= 5,962   trend_pct from     -20.0% to      20.0%
  down    n=16,262   trend_pct from    -100.0% to     -20.0%

  -> the boundaries are exactly +20% and -20%, i.e. the +/-20% rule.
  -> the paper's stated +/-10% would place every 'stable' page between -20% and -10%
     into 'down'. That is 1,935 rows in this slice alone that would change class.

FINDING A - 'growth-to-decline ratio by freshness window' (paper claims 7.88:1 at 31-90d)
trend_direction    up   down  ratio  lift_vs_base
freshness_band                                   
0-30             3168  10473   0.30          1.11
31-90              38    103   0.37          1.37
91-180           1155   5604   0.21          0.78
181-360            27     79   0.34          

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

**Both numbers, same model, same features, same seed.** This is Question B2 turned on my own work — I do
not get to raise the split objection against the paper without showing what it costs me.

| Split | ROC-AUC | precision@50 | What it measures |
|---|---|---|---|
| Random 80/20 rows | **0.777** | 0.920 | Partly: which client this page belongs to |
| **Grouped, clients held out** | **0.624** | 0.780 | Whether it works on a client never seen |

**The gap is 0.152 AUC, and it is the honest cost of my headline.** Put in the only terms that matter:
my model's genuine skill above chance on the grouped split is 0.624 − 0.500 = **0.124**. The inflation a
random split would have handed me is **0.152**. **The memorisation is larger than the signal.** Had I
reported the random-split number, more than half of what I showed a reader would have been the model
recognising which client a page belongs to.

Every number in ML-08 and in my capstone report uses the grouped split. This cell exists so a reader can
see what I gave up rather than take my word for it — and because I raised exactly this objection against
someone else's paper one section ago.

**A time-aware split is not available and I am not going to fake one.** ML-04 established that this file
has no date column — every time field is a relative offset from an unstated snapshot. Train-on-past /
test-on-future needs `fact_content_daily_performance` from the warehouse release (`report_date` 2025-01-27
→ 2026-06-30). Since my label is a 30-day-vs-30-day change measured *inside* the feature window, a genuine
time split would also change the task from **concurrent decline detection** to **forecasting** — a
different, harder problem, and one I have deliberately not claimed to solve.

**Where the memorisation actually lives.** The per-client breakdown in ML-08 showed out-of-fold AUC
ranging from 0.506 to 0.770 across clients with ≥200 pages. A random split hides that entirely behind one
portfolio-wide number; the grouped split is what makes the variation visible, and the variation is the
operationally important part.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one - typing sentences here breaks Run All.
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.base import clone

# The ML-08 feature matrix, rebuilt exactly (contract fields, flags, log1p on the heavy tails).
FEATURES = [
    "search_volume", "competition", "competition_level", "cpc",
    "word_count", "char_count", "word_count_tier", "char_count_tier",
    "content_type", "main_intent",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions",
    "content_age_days", "age_tier", "age_tier_order", "days_since_last_update", "freshness_tier",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
    "impression_tier", "position_tier",
]
FLAG_COLS = ["search_volume", "competition", "cpc", "word_count", "char_count", "avg_position"]
COUNT_COLS = ["impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
              "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d", "search_volume"]


def build_matrix(extra_cols=()):
    """The ML-08 matrix. extra_cols lets me deliberately inject leaks in Section 3."""
    cols = FEATURES + list(extra_cols)
    X = df[cols].copy()
    X["avg_position"] = X["avg_position"].replace(0, np.nan)
    for c in FLAG_COLS:
        X[f"has_{c}"] = X[c].notna().astype(int)
    X[COUNT_COLS] = np.log1p(X[COUNT_COLS])
    cat = [c for c in cols if not is_numeric_dtype(df[c])]
    num = [c for c in X.columns if c not in cat]
    return X, num, cat


def make_pipe(num, cat, model="logistic"):
    steps = [("i", SimpleImputer(strategy="median"))]
    if model == "logistic":
        steps.append(("s", StandardScaler()))
        clf = LogisticRegression(max_iter=2000, C=1.0, random_state=RANDOM_STATE)
    else:
        clf = HistGradientBoostingClassifier(max_depth=4, learning_rate=0.06, max_iter=300,
                                             random_state=RANDOM_STATE)
    pre = ColumnTransformer([
        ("num", Pipeline(steps), num),
        ("cat", Pipeline([("i", SimpleImputer(strategy="constant", fill_value="__missing__")),
                          ("o", OneHotEncoder(handle_unknown="ignore", min_frequency=20))]), cat),
    ])
    return Pipeline([("pre", pre), ("clf", clone(clf))])


X, NUM, CAT = build_matrix()
folds = list(GroupKFold(n_splits=5).split(X, y, groups))

# --- BEFORE: the random split I am NOT reporting -------------------------
tr_r, te_r = train_test_split(np.arange(len(df)), test_size=0.2, random_state=RANDOM_STATE, stratify=y)
p_rand = make_pipe(NUM, CAT, "boost").fit(X.iloc[tr_r], y[tr_r]).predict_proba(X.iloc[te_r])[:, 1]

# --- AFTER: the grouped split every reported number uses ------------------
tr_g, te_g = folds[1]
p_grp = make_pipe(NUM, CAT, "boost").fit(X.iloc[tr_g], y[tr_g]).predict_proba(X.iloc[te_g])[:, 1]

auc_r, auc_g = roc_auc(y[te_r], p_rand), roc_auc(y[te_g], p_grp)
print("Same model, same features, same seed - only the split changes:")
print(f"  {'random 80/20 rows':<28} AUC {auc_r:.4f}   p@50 {precision_at_k(y[te_r], p_rand, 50):.3f}"
      f"   ({len(te_r):,} test rows)")
print(f"  {'grouped, clients held out':<28} AUC {auc_g:.4f}   p@50 {precision_at_k(y[te_g], p_grp, 50):.3f}"
      f"   ({len(te_g):,} test rows)")
print(f"\n  inflation the random split would have handed me: {auc_r - auc_g:+.4f} AUC "
      f"({(auc_r - auc_g) / (auc_g - 0.5) * 100:.0f}% of my real skill above chance)")
print("  -> that gap is client memorisation. Reported so the cost of the honest split is visible.")

# --- why a time split is impossible here, asserted -----------------------
import re
DATE_TOKENS = {"date", "datetime", "timestamp", "ts", "day", "month", "year", "week"}
date_like = [c for c in df.columns if DATE_TOKENS & set(re.split(r"_+", c.lower()))]
print(f"\nabsolute date columns available for a time split: {date_like or 'none'}")
assert date_like == [], "a date column exists - revisit the claim that a time split is impossible"
print("  -> no time-aware split is constructible from this file; it needs the warehouse release.")

# --- per-client variation the random split would have hidden -------------
oof = np.full(len(df), np.nan)
for tr, te in folds:
    oof[te] = make_pipe(NUM, CAT).fit(X.iloc[tr], y[tr]).predict_proba(X.iloc[te])[:, 1]
per_client = [roc_auc(y[g.index], oof[g.index]) for _, g in df.groupby("client_id") if len(g) >= 200]
per_client = [v for v in per_client if v == v]
print(f"\nper-client out-of-fold AUC (>=200 pages): {min(per_client):.3f} to {max(per_client):.3f}, "
      f"median {np.median(per_client):.3f}, n={len(per_client)} clients")
print("  -> one portfolio-wide number hides this spread entirely.")

Same model, same features, same seed - only the split changes:
  random 80/20 rows            AUC 0.7765   p@50 0.920   (6,000 test rows)
  grouped, clients held out    AUC 0.6244   p@50 0.780   (5,731 test rows)

  inflation the random split would have handed me: +0.1521 AUC (122% of my real skill above chance)
  -> that gap is client memorisation. Reported so the cost of the honest split is visible.

absolute date columns available for a time split: none
  -> no time-aware split is constructible from this file; it needs the warehouse release.



per-client out-of-fold AUC (>=200 pages): 0.506 to 0.770, median 0.642, n=21 clients
  -> one portfolio-wide number hides this spread entirely.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Running the full taxonomy from `skills/hunting-leakage-and-validating` against the **38-column ML-08
matrix that produced my shipped result** — not against the raw file, which is what ML-05 audited.

### The attack checklist

| Check | Result |
|---|---|
| Timeline: features knowable before the label window | **Not fully satisfiable, and disclosed.** See below. |
| No label-derived or sibling columns | **Pass** — asserted in code; the four forbidden fields are absent |
| No product flags / existing-system scores | **Pass** — `provider_used`, `model_used` excluded in ML-04 |
| Population selection free of outcome-window info | **Pass, with a disclosed survivorship filter** |
| Split grouped by the repeating entity | **Pass** — GroupKFold on `client_id`, zero overlap asserted |
| Base rate printed next to every metric | **Pass** — 0.542 appears beside every precision figure |
| Top feature sanity-checked | **Pass** — 0.119 AUC drop, strong but ordinary |
| Metrics out-of-fold, never in-sample | **Pass** for the model; the baseline is in-sample and disclosed |
| Sealed/holdout receipts committed | **N/A — no sealed claim is made** |

### 1. Label-derived features — and the test that proves my harness works

The skill's verification step: *deliberately add a leaky feature and watch the score jump toward 1.0 — if
it doesn't, your test harness itself is broken.* I ran it, and it produced a result I did not plan for.

| Feature set | Out-of-fold AUC | precision@50 |
|---|---|---|
| Clean ML-08 matrix | **0.677** | 0.880 |
| + the two forbidden columns, as **raw counts** | **0.920** | 1.000 |
| + the two forbidden columns, in **log space** | **~1.000** | 1.000 |

**The harness confesses when a leak is present, which is what makes its "no leak" verdict on the clean
matrix worth anything.** That is the receipt for the rest of this notebook.

**The unplanned finding is the second and third rows.** The *same two columns* score 0.92 raw and ~1.00
logged. The reason is that the label is a threshold on a **ratio** — `(last − prev)/prev < −0.20` — and a
ratio is linear in logs but not in raw counts. A linear model can only fully exploit the leak when the
representation matches the label's functional form.

**So "is this column a leak?" is the wrong question.** Leakage is a property of *(column, representation,
model)*, not of a column alone. A screen that checks columns before deciding how they will be transformed
can under-state a leak by **0.08 AUC** — which is precisely the size of gap that gets waved through as
"the model is just a bit good". This also sharpens ML-05's original finding: that audit caught the pair by
testing the *reconstruction formula* rather than the columns, and this is why that was the right move.

### 2. Future / overlapping windows — the one check I cannot fully pass

**This is the honest weak point of the whole project, and it is structural rather than fixable.** The
label is a 30-day-vs-previous-30-day impression change, i.e. it lives in days 1–60 of the window. Several
features (`impressions_90d`, `clicks_90d`, `days_with_impressions`, …) are 90-day aggregates that
**contain those same 60 days**.

By the strict rule — *a feature summed over a window containing the label's window already knows the
outcome* — those features are contaminated. Three things keep this from being fatal, and I would rather
state all three than pretend the check passed:

1. **It is a magnitude overlap, not an identity.** ML-05 measured that the label is exactly reconstructible
   from the two 30-day columns (1.0000) but the 90-day totals do not reconstruct it — a 90-day sum tells
   you *how much* traffic there was, not *which direction it moved*. The measured single-feature AUC of
   `impressions_90d` is 0.585, not 0.99.
2. **The task is honestly framed as concurrent, not predictive.** I claim "this page resembles the pages
   measured as declining in this window", never "this page will decline". Under that framing an
   overlapping window is a description of the same period, not a peek at the future.
3. **The fix requires data this file does not contain.** A clean version needs features from days 61–90
   only, with the label from days 1–30 — reconstructible from `fact_content_daily_performance` in the
   warehouse release, and impossible here because the 30-day columns do not tile the 90-day window (they
   cover days 1–60; ML-04 measured the sum matching in only 8.7% of rows).

**This limitation belongs in the paper, and it is in it** (capstone report, Section 4 and Limitation 2).

### 3. Decision-derived features (product flags)

**Pass.** `provider_used` (71.5% missing) and `model_used` (19.1% missing) name which internal pipeline
generated a page. Using them would teach the model *which tool made this* rather than *is this page
decaying* — the circular result the taxonomy warns about. Both were excluded in ML-04, before any model
existed. The FlyRank `health_score` and the optimization flags described in the paper are the same species
of feature; neither is in this dataset, but if a future release includes them they belong in the
**baseline-to-beat**, never in the inputs.

### 4. Population selection

**Pass, with a disclosed filter.** The file was pre-filtered upstream: minimum `content_age_days` is
exactly 90, and every row has `impressions_90d > 0` and `sessions_90d > 0`. That is a survivorship cut —
pages that died completely are absent. It does **not** use outcome-window information (the filter is on
existence and age, not on the trend), so it is not leakage; but it does bound every claim to *mature pages
that still get some traffic*, which is stated wherever a portfolio-level rate appears.

### 5. Single-feature discrimination sweep

Re-run on the final matrix below. No feature approaches the alarm zone; the strongest is
`impressions_90d`. The two columns that *would* light up are excluded by contract.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one - typing sentences here breaks Run All.
# --- CHECK 1: no forbidden column is in the final matrix ------------------
FORBIDDEN = {"trend_direction", "trend_pct", "impressions_last_30d", "impressions_prev_30d",
             "is_declining_label", "clicks_last_30d", "clicks_prev_30d",
             "sessions_last_30d", "sessions_prev_30d", "provider_used", "model_used",
             "content_id", "client_id"}
present = FORBIDDEN & set(X.columns)
print(f"final matrix: {X.shape[1]} columns | forbidden columns present: {sorted(present) or 'none'}")
assert not present, "a forbidden column reached the final feature set"

# --- CHECK 2: the harness-validation test (inject a known leak) -----------
print("\n" + "=" * 78)
print("HARNESS VALIDATION - deliberately add the known leak back and watch the score confess")
LEAK = ("impressions_last_30d", "impressions_prev_30d")

# (a) injected as RAW counts
X_raw, NUM_R, CAT_R = build_matrix(extra_cols=LEAK)
oof_raw = np.full(len(df), np.nan)
for tr, te in folds:
    oof_raw[te] = make_pipe(NUM_R, CAT_R).fit(X_raw.iloc[tr], y[tr]).predict_proba(X_raw.iloc[te])[:, 1]

# (b) injected in LOG space, where the label's ratio rule becomes linear:
#     (last-prev)/prev < -0.20  <=>  log(last) - log(prev) < log(0.8)
X_log, NUM_G, CAT_G = build_matrix(extra_cols=LEAK)
X_log[list(LEAK)] = np.log1p(X_log[list(LEAK)])
oof_log = np.full(len(df), np.nan)
for tr, te in folds:
    oof_log[te] = make_pipe(NUM_G, CAT_G).fit(X_log.iloc[tr], y[tr]).predict_proba(X_log.iloc[te])[:, 1]

auc_clean, auc_raw, auc_log = roc_auc(y, oof), roc_auc(y, oof_raw), roc_auc(y, oof_log)
print(f"  clean ML-08 feature set           out-of-fold AUC {auc_clean:.4f}   "
      f"p@50 {precision_at_k(y, oof, 50):.3f}")
print(f"  + the leak, as RAW counts         out-of-fold AUC {auc_raw:.4f}   "
      f"p@50 {precision_at_k(y, oof_raw, 50):.3f}   ({auc_raw - auc_clean:+.4f})")
print(f"  + the leak, in LOG space          out-of-fold AUC {auc_log:.4f}   "
      f"p@50 {precision_at_k(y, oof_log, 50):.3f}   ({auc_log - auc_clean:+.4f})")
assert auc_log > 0.98, "the harness FAILED to detect a known leak - the test setup is broken"
assert auc_clean < 0.80, "the clean feature set scores suspiciously high - re-audit"
print("\n  -> the harness detects a leak when one exists, so its 'no leak' verdict on the clean")
print("     feature set is evidence rather than an assumption. This is the receipt.")
print("\n  AND AN UNPLANNED FINDING: the SAME leak scores 0.92 raw but ~1.00 in log space.")
print("  The label is a threshold on a RATIO, which is linear in logs and not in raw counts,")
print("  so a linear model can only fully exploit the leak when the representation matches the")
print("  label's functional form. A leak is therefore not a fixed property of a column - it is a")
print("  property of (column, representation, model). Screening features without considering how")
print("  they will be transformed can under-state a leak by 0.08 AUC, which is exactly the size of")
print("  gap that gets waved through as 'the model is just a bit good'.")

# --- CHECK 3: single-feature discrimination sweep on the FINAL matrix -----
print("\n" + "=" * 78)
print("SINGLE-FEATURE SWEEP on the final matrix (|AUC - 0.5| = standalone discriminative power)")
rows = []
for c in NUM:
    v = X[c]
    if v.notna().sum() > 0 and v.nunique(dropna=True) > 1:
        a = roc_auc(y, v.fillna(v.median()))
        rows.append((c, a, abs(a - 0.5)))
sweep = pd.DataFrame(rows, columns=["feature", "auc", "deviation"]).sort_values("deviation", ascending=False)
print(sweep.head(8).round(4).to_string(index=False))
print(f"\n  strongest single feature: {sweep.iloc[0]['feature']} at AUC {sweep.iloc[0]['auc']:.4f}")
print("  for contrast, the EXCLUDED columns (never used - measured here for the audit only):")
for c in ("trend_pct", "impressions_prev_30d", "impressions_last_30d"):
    v = df[c].fillna(df[c].median())
    print(f"    {c:<22} AUC {roc_auc(y, v):.4f}")
assert sweep.iloc[0]["deviation"] < 0.25, "a single feature separates the classes too well - investigate"
print("  -> nothing in the matrix is near the alarm zone; the two that are, are excluded.")

# --- CHECK 4: the overlapping-window problem, quantified ------------------
print("\n" + "=" * 78)
print("THE WINDOW OVERLAP I CANNOT FULLY CLEAR (disclosed, not hidden)")
tiles = (df["impressions_last_30d"] + df["impressions_prev_30d"] == df["impressions_90d"]).mean()
print(f"  the label's window (days 1-60) sits INSIDE the 90-day feature window")
print(f"  30-day columns tile the 90-day total in only {tiles:.1%} of rows -> days 61-90 are not separable")
print(f"  but a 90-day SUM does not reconstruct a DIRECTION: impressions_90d alone scores "
      f"AUC {roc_auc(y, df['impressions_90d']):.4f}")
print("  -> magnitude overlap, not identity. Framed as concurrent detection, never as a forecast.")

# --- CHECK 5: population selection ---------------------------------------
print("\n" + "=" * 78)
print("POPULATION SELECTION (a survivorship filter, disclosed - not outcome-window information)")
print(f"  min content_age_days {df['content_age_days'].min()} -> pages younger than 90 days removed upstream")
print(f"  rows with impressions_90d == 0: {(df['impressions_90d'] == 0).sum()} | "
      f"sessions_90d == 0: {(df['sessions_90d'] == 0).sum()}")
print("  -> the filter keys on existence and age, NOT on the trend, so it does not leak the outcome.")
print("     It does bound every claim to 'mature pages that still get some traffic'.")

# --- CHECK 6: no sealed claim is being made ------------------------------
print("\n" + "=" * 78)
print("SEALED / HOLDOUT CLAIM: none made.")
print("  Cross-validation on held-out clients is an honest out-of-sample estimate, NOT a sealed test:")
print("  every fold was visible to me while working, and I chose the shipped configuration after")
print("  seeing fold results. No metrics file in this repo claims a blind evaluation.")

final matrix: 38 columns | forbidden columns present: none

HARNESS VALIDATION - deliberately add the known leak back and watch the score confess


  clean ML-08 feature set           out-of-fold AUC 0.6774   p@50 0.880
  + the leak, as RAW counts         out-of-fold AUC 0.9199   p@50 1.000   (+0.2425)
  + the leak, in LOG space          out-of-fold AUC 0.9996   p@50 1.000   (+0.3221)

  -> the harness detects a leak when one exists, so its 'no leak' verdict on the clean
     feature set is evidence rather than an assumption. This is the receipt.

  AND AN UNPLANNED FINDING: the SAME leak scores 0.92 raw but ~1.00 in log space.
  The label is a threshold on a RATIO, which is linear in logs and not in raw counts,
  so a linear model can only fully exploit the leak when the representation matches the
  label's functional form. A leak is therefore not a fixed property of a column - it is a
  property of (column, representation, model). Screening features without considering how
  they will be transformed can under-state a leak by 0.08 AUC, which is exactly the size of
  gap that gets waved through as 'the model is just a bit good'.



              feature    auc  deviation
     content_age_days 0.4085     0.0915
       age_tier_order 0.4149     0.0851
      impressions_90d 0.5845     0.0845
days_with_impressions 0.5794     0.0794
        search_volume 0.4396     0.0604
     has_avg_position 0.5433     0.0433
      has_competition 0.5403     0.0403
              has_cpc 0.5403     0.0403

  strongest single feature: content_age_days at AUC 0.4085
  for contrast, the EXCLUDED columns (never used - measured here for the audit only):
    trend_pct              AUC 0.0449
    impressions_prev_30d   AUC 0.6214
    impressions_last_30d   AUC 0.4857
  -> nothing in the matrix is near the alarm zone; the two that are, are excluded.

THE WINDOW OVERLAP I CANNOT FULLY CLEAR (disclosed, not hidden)
  the label's window (days 1-60) sits INSIDE the 90-day feature window
  30-day columns tile the 90-day total in only 8.7% of rows -> days 61-90 are not separable
  but a 90-day SUM does not reconstruct a DIRECTION: impressions_90d 

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional,
decision-support.*

### My boldest sentence, as I was tempted to write it

> ~~"My model identifies declining content with 90% precision, so refreshing the pages it selects will
> recover lost organic traffic."~~

Four separate failures in one sentence, each mapped to the claim ladder:

| Fragment | What is wrong | Evidence I actually have |
|---|---|---|
| "identifies declining content" | Implies detection of a real-world state | A **proxy label**: a −20% threshold on a 30-day impression ratio |
| "with 90% precision" | A bare number with no base rate, no K, no split | 0.900 **at K=50**, against a **0.542 base rate**, out-of-fold |
| "refreshing ... will recover" | **Causal claim.** No intervention exists in this dataset | Zero refreshes were performed or measured |
| "lost organic traffic" | Implies a revenue/traffic outcome | Impressions only — no clicks, sessions or revenue outcome measured |

### The rewrite I stand behind

> **On a client-held-out split of 30,000 mature, still-trafficked content items from 32 pseudonymized
> clients, a ranking that combines a five-condition rule with a logistic model placed 45 of its top 50
> pages on items measured as declining (precision@50 = 0.900, against a 0.542 base rate; bootstrap 95%
> interval [0.820, 0.980]). "Declining" here means a measured impression drop of more than 20% between two
> consecutive 30-day windows — a proxy defined in the data, not an observed editorial or revenue outcome.
> The ranking is decision-support for triage: it indicates which pages an editor may find worth opening
> first in this portfolio and window. It does not establish that these pages will continue to decline, and
> nothing in this dataset supports a claim that refreshing them recovers traffic.**

Longer, and every clause is doing load-bearing work: **population** (mature, still-trafficked, 32 clients),
**split** (client-held-out), **metric with its K and base rate**, **uncertainty**, **the label's
definition as a proxy**, **the decision it supports**, and **the two claims explicitly not being made**.

### Three more of my own sentences, corrected

| Tempting version | Honest version |
|---|---|
| "The model beats the baseline." | "The shipped ranking scored higher at K=20–100; **the baseline scored higher at K=200**, and the per-fold floor favours a model I did not ship." |
| "Days with impressions is the key driver of decline." | "`impressions_90d` showed the largest permutation importance (0.119 AUC). Because the count features are collinear, **individual contributions are not separable**, and this is association within one window, not a driver." |
| "The model fixed the baseline's ordering problem." | "The shipped ranking's precision falls monotonically with K where the rule's rises — the specific defect it was built to address. **On one dataset and one 90-day window**; the p@50 intervals overlap." |

### The rule I am taking forward

Every claim now has to name **four** things before I will write it down: the **population** it applies to,
the **split** it was measured on, the **base rate** beside the metric, and the **decision** it supports.
If any of the four is missing, the sentence is not yet a finding.

**And the reciprocal check, since I spent Section 1 auditing someone else's paper:** the two objections I
raised there — a metric without its base rate, and an ungrouped split across a repeating entity — are the
two I am most at risk of committing myself. Section 2 is my answer to the second. The base rate appears
beside every precision figure I report, which is my answer to the first.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one - typing sentences here breaks Run All.
# Every number the rewritten claim asserts, verified from the committed ML-08 receipt.
receipt = json.loads(Path("work/outputs/model_metrics.json").read_text())
shipped = receipt["systems"][receipt["shipped"]]

print("VERIFYING EVERY CLAUSE OF THE REWRITTEN CLAIM")
print(f"  population   : {len(df):,} rows, {df['client_id'].nunique()} clients, "
      f"min content_age_days {df['content_age_days'].min()}, all with impressions_90d > 0")
print(f"  split        : {receipt['split']}")
print(f"  metric       : precision@50 = {shipped['p@50']}  at K=50")
print(f"  base rate    : {receipt['base_rate']}  (majority class {max(BASE_RATE, 1-BASE_RATE):.1%})")
print(f"  slots        : {shipped['p@50'] * 50:.0f} of 50 pages measured as declining "
      f"(random triage: {BASE_RATE * 50:.0f})")
print(f"  label meaning: trend_pct < -20% between two consecutive 30-day windows")
print(f"  uncertainty  : bootstrap 95% CI [0.820, 0.980] (ML-08, Section 3)")

# The two claims the sentence explicitly does NOT make - asserted, not promised.
print("\nWHAT THE SENTENCE REFUSES TO CLAIM, and why it must:")
print(f"  no intervention exists: the file has no refresh-event column, so no before/after is measurable")
INTERVENTION_HINTS = ("refresh", "updated_at", "intervention", "treatment", "action_taken")
found = [c for c in df.columns if any(h in c.lower() for h in INTERVENTION_HINTS)]
print(f"    columns recording an action taken on a page: {found or 'none'}")
assert not found, "an intervention column exists - the causal framing would need revisiting"
print(f"  no outcome beyond impressions: the label reads impressions only, not clicks/sessions/revenue")
print(f"    correlation between the label and a clicks-based version of the same ratio: "
      f"{np.corrcoef(y, (df['clicks_last_30d'] < df['clicks_prev_30d']).astype(int))[0,1]:.3f} "
      f"-> a different metric would give a different label")

# The honest counter-evidence the claim has to survive.
print("\nCOUNTER-EVIDENCE CARRIED IN THE SAME BREATH (from the ML-08 receipt):")
print(f"  baseline still wins at K=200 : rule {receipt['systems']['baseline_rule']['p@200']} vs "
      f"shipped {shipped['p@200']}")
pf = receipt["per_fold_precision_at_50"]
print(f"  per-fold floor favours a model I did NOT ship: "
      f"boosting {min(pf['hist_gradient_boost']):.3f} vs shipped {min(pf['hybrid']):.3f}")
print(f"  known failure mode           : {receipt['known_failure_mode']}")
print(f"  portfolio coverage           : top-50 spans {receipt['top50_client_concentration']['distinct_clients']} "
      f"clients, largest {receipt['top50_client_concentration']['largest_client_share']:.0%} "
      f"(worse than the rule's 44%)")
print("\n  -> a claim that omits any of these is not a shorter claim, it is a different one.")

VERIFYING EVERY CLAUSE OF THE REWRITTEN CLAIM
  population   : 30,000 rows, 32 clients, min content_age_days 90, all with impressions_90d > 0
  split        : GroupKFold(5) on client_id - zero client overlap, asserted
  metric       : precision@50 = 0.9  at K=50
  base rate    : 0.5421  (majority class 54.2%)
  slots        : 45 of 50 pages measured as declining (random triage: 27)
  label meaning: trend_pct < -20% between two consecutive 30-day windows
  uncertainty  : bootstrap 95% CI [0.820, 0.980] (ML-08, Section 3)

WHAT THE SENTENCE REFUSES TO CLAIM, and why it must:
  no intervention exists: the file has no refresh-event column, so no before/after is measurable
    columns recording an action taken on a page: none
  no outcome beyond impressions: the label reads impressions only, not clicks/sessions/revenue
    correlation between the label and a clicks-based version of the same ratio: 0.137 -> a different metric would give a different label

COUNTER-EVIDENCE CARRIED IN THE SAME

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled - markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] The paper critique is checkable in code, not a matter of opinion - and credits what it does well
- [x] I ran the same attacks on my own model that I raised against the paper
- [ ] Committed to my repo under `work/notebooks/` - then submit your repo URL on the card. Done.